<a href="https://colab.research.google.com/drive/1p9aAUoFpKcrwR2ISte6Yj0mnjviAsjPs?usp=sharing" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 1: Russian noun morphology with HFST

Author: Iļja Žuravļovs  
Course: Fundamentals of NLP  

Chosen fragment:
- language: Russian
- part of morphology: feminine nouns of the 1st declension
- current nouns: `берёз`, `книг`, `башн`, `вишн`, `кухн`

Implemented in this draft:
- singular and plural noun forms
- insertion rule
- substitution rule
- removal rule
- one more complex case: `кухн+N+Pl+Gen -> кухонь`


In [ ]:
# Install HFST first, if needed
!pip install hfst

In [ ]:
!wget https://raw.githubusercontent.com/TiGReX423/NLP_assignment_1/refs/heads/main/russian_feminine_nouns.lexc

In [ ]:
import hfst
from hfst import HfstTransducer, compose

generator = hfst.compile_lexc_file('russian_feminine_nouns.lexc')
print(generator)

If HFST cannot read the lexc file, use the code below to create a temporary file with the same lexicon.

In [ ]:
from pathlib import Path
import hfst

lexc_text = r"""
Multichar_Symbols
        +N
        +Sg
        +Pl
        +Nom
        +Gen
        +Dat
        +Acc
        +Ins
        +Prep

LEXICON Root
        Nouns ;

LEXICON Nouns
берёза   NSimpleA ;
книга    NSimpleA ;
башня    NSimpleJa ;
вишня    NSimpleJa ;
кухня    NComplexJa ;

LEXICON NSimpleA
+N:^   SimpleA ;

LEXICON NSimpleJa
+N:^   SimpleJa ;

LEXICON NComplexJa
+N:^   ComplexJa ;

LEXICON SimpleA
+Sg+Nom:а    # ;
+Sg+Gen:ы    # ;
+Sg+Dat:е    # ;
+Sg+Acc:у    # ;
+Sg+Ins:ой   # ;
+Sg+Prep:е   # ;
+Pl+Nom:ы    # ;
+Pl+Gen:Z    # ;
+Pl+Dat:ам   # ;
+Pl+Acc:ы    # ;
+Pl+Ins:ами  # ;
+Pl+Prep:ах  # ;

LEXICON SimpleJa
+Sg+Nom:я    # ;
+Sg+Gen:и    # ;
+Sg+Dat:е    # ;
+Sg+Acc:ю    # ;
+Sg+Ins:ей   # ;
+Sg+Prep:е   # ;
+Pl+Nom:и    # ;
+Pl+Gen:F    # ;
+Pl+Dat:ям   # ;
+Pl+Acc:и    # ;
+Pl+Ins:ями  # ;
+Pl+Prep:ях  # ;

LEXICON ComplexJa
+Sg+Nom:я    # ;
+Sg+Gen:и    # ;
+Sg+Dat:е    # ;
+Sg+Acc:ю    # ;
+Sg+Ins:ей   # ;
+Sg+Prep:е   # ;
+Pl+Nom:и    # ;
+Pl+Gen:ьK   # ;
+Pl+Dat:ям   # ;
+Pl+Acc:и    # ;
+Pl+Ins:ями  # ;
+Pl+Prep:ях  # ;

END
"""

Path("temp.lexc").write_text(lexc_text, encoding="utf-8", newline="\n")
generator = hfst.compile_lexc_file("temp.lexc")
print(generator)

In [ ]:
print(generator.lookup('книга+N+Pl+Nom'))
print(generator.lookup('башня+N+Pl+Gen'))
print(generator.lookup('кухня+N+Pl+Gen'))

## FST cascading

Rules in this draft:
- insertion of `е` in genitive plural after `шн`
- insertion of `о` in the more complex form `кухонь`
- substitution `ы -> и` after `г | к | х | ж | ш | щ | ч | ц`
- deletion of technical markers


In [89]:
DeleteA = hfst.regex('а -> 0 || _ "^"')
DeleteYa = hfst.regex('я -> 0 || _ "^"')

SpellingRule = hfst.regex('ы -> и || [г | к | х | ж | ш | щ | ч | ц] "^" _')
FleetingE = hfst.regex('[..] -> е || ш _ н "^" F')
KitchenO = hfst.regex('[..] -> о || х _ н "^" ь K')

MarkerCleanup = hfst.regex('[F | K | Z] -> 0')
CaretCleanup = hfst.regex('"^" -> 0')

cascade = compose((
    generator,
    DeleteA,
    DeleteYa,
    SpellingRule,
    FleetingE,
    KitchenO,
    MarkerCleanup,
    CaretCleanup
))

cascade.remove_epsilons()

In [ ]:
tests = [
    'берёза+N+Sg+Nom',
    'берёза+N+Pl+Gen',
    'книга+N+Sg+Gen',
    'книга+N+Pl+Nom',
    'башня+N+Sg+Ins',
    'башня+N+Pl+Gen',
    'вишня+N+Pl+Gen',
    'кухня+N+Pl+Gen'
]

for item in tests:
    print(item, '->', cascade.lookup(item)[0][0].replace("@_EPSILON_SYMBOL_@", ""))

## Analysis mode


In [ ]:
analyzer = HfstTransducer(cascade)
analyzer.invert()
analyzer.convert(hfst.ImplementationType.HFST_OL_TYPE)

for word in ['книги', 'башен', 'вишен', 'кухонь']:
    print(word, '->', analyzer.lookup(word))

## FST visualization

In [80]:
import sys

def hfst2png(transducer, png_name):
    # Write the FST to a file using the att format
    f = open("a.att", "w", encoding="utf-8")
    transducer.minimize()
    transducer.write_att(f, False)

    # Convert the FST file to the dot format
    # dot format is used by graphviz library for graph visualization
    f = open("a.att", "r", encoding="utf-8")
    with open("graph.dot", "w", encoding="utf-8") as out_f:
        out_f.write('digraph G { rankdir="LR"\n')
        out_f.write('node [fontname="Tahoma",shape=circle,fontsize=14,fixedsize=true,fillcolor="grey",style=filled]\n')
        out_f.write('edge [fontname="FreeMono",fontsize=14]\n')
        for line in f.readlines():
            line = line.strip()
            row = line.split('\t')
            if len(row) >= 4:
                out_f.write('%s [label="%s"];\n' % (row[0], row[0]))
                out_f.write('%s -> %s [label="%s:%s"];\n' % (row[0], row[1], row[2], row[3]))
            elif len(row) == 1: # Final state
                out_f.write('%s [label="%s",shape=doublecircle];\n' % (row[0], row[0]))
        out_f.write('}')

    # Call graphviz dot function to generate a png file from dot file
    !dot -Tpng graph.dot > $png_name

In [81]:
# Call the transformation to png on the HFSTTransducer "cascade"
hfst2png(cascade,'image.png')

In [ ]:
# Display image in notebook
from IPython.display import Image
Image('image.png')

In [ ]:
for noun in ["берёза", "книга", "башня", "вишня", "кухня"]:
    for number in ["Sg", "Pl"]:
        for form in ["Nom", "Gen", "Dat", "Acc", "Ins", "Prep"]:
            query = noun + "+N+" + number + "+" + form
            result = cascade.lookup(query)
            if result:
                print(query + " -> " + result[0][0].replace("@_EPSILON_SYMBOL_@", ""))
            else:
                print(query + " -> NO RESULT")